## MD engine optimization and streaming/fileio performance analysis

In [1]:
# import libararies for plotting and some analysis
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
# current working directory
os.getcwd()

'/scratch/athiru12/IMDv3-performance-tests'

In [2]:
from os import popen, makedirs, system, walk
from os.path import join, isfile, isdir, basename, dirname, exists
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import numpy as np
import pandas as pd
import os
from os.path import join
from cycler import cycler
import random
from matplotlib.ticker import AutoMinorLocator
from matplotlib.ticker import FormatStrFormatter
from matplotlib.ticker import NullFormatter
from matplotlib.ticker import FixedFormatter
from matplotlib.ticker import LogFormatterSciNotation
from matplotlib.ticker import FixedLocator
from matplotlib.ticker import MultipleLocator
from matplotlib.ticker import LogLocator
from matplotlib.ticker import MaxNLocator
from mpl_toolkits import mplot3d
from sklearn.metrics import pairwise_distances
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import pairwise_distances
analysis_folder = "output"
graphs_folder = "output/graphs"
makedirs(graphs_folder, exist_ok=True)
#Options
from cycler import cycler
fig_width_pt = 246 #320 #/2 or 510   # Get this from LaTeX using \showthe\columnwidth
inches_per_pt = 1.0/72.27               # Convert pt to inch
golden_mean = (np.sqrt(5)-1.0)/2.0         # Aesthetic ratio
fig_width = fig_width_pt*inches_per_pt  # width in inches
fig_height = fig_width*golden_mean      # height in inches
fig_size =  [fig_width,fig_height]
params = {'backend': 'ps',
          #'axes.labelsize': 10,
          'font.size': 9,
          'font.weight': "normal",
          "font.family": "serif",
          "font.serif": ['Computer Modern Roman'],
          'legend.fontsize': 6,
          'xtick.labelsize': 7.5,
          'ytick.labelsize': 7.5,
          'xtick.major.width': 0.4,
          'xtick.minor.width': 0.3,
          'ytick.major.width': 0.4,
          'ytick.minor.width': 0.3,
          'text.usetex': True,
          'axes.linewidth': 0.5,
          'axes.prop_cycle': cycler(color='bgrcmyk'),
          'figure.figsize': fig_size
          }
plt.rcParams.update(params)
# colors = ["blue", "green", "red", "cyan", "magenta", "yellow", "blue", "green", "red"]
colors_blue = ['#08306b', '#2171b5', '#6baed6', '#c6dbef']  # Dark blue to light blue
colors_green = ['#00441b', '#238b45', '#74c476', '#c7e9c0'] # Dark green to light green
colors_red = ['#67000d', '#cb181d', '#fb6a4a', '#fcbba1']   # Dark red to light red
colors_orange = ['#ff7f00', '#ffbb78', '#ff9900', '#ffcc99'] # Dark orange to light orange
colors_purple = ['#4a1486', '#6a51a3', '#9e9ac8', '#dadaeb'] # Dark purple to light purple
colors_teal = ['#004d4d', '#008080', '#66b2b2', '#cce5e5'] # Dark teal to light teal
colors = ["blue", "green", "red", "cyan", "magenta", "yellow", "blue", "green", "red"]

In [3]:
# Read all engine data into a single dataframe with MDengine labels
common_columns = [
    "JobSubmitDateTime",
    "JobStartDateTime",
    "MDStartDateTime",
    "MDEndDateTime",
    "ClusterName",
    "nNodereq",
    "NodeNameList",
    "nGPUreq",
    "GPUIDList",
    "GPUtypereq",
    "nCPUreq",
    "CPUIDList",
    "SimType",
    "RunPurpose",
    "RunType",
    "nNodeused",
    "NodeNameused",
    "nGPUused",
    "GPUtypeused",
    "GPUIDsused",
    "nCPUused",
    "CPUIDsused",
    "MPItasksperNode",
    "OpenMPthreadsperNode",
    "FileIOfreqLog",
    "FileIOfreqBox",
    "FileIOfreqPositions",
    "FileIOfreqVelocities",
    "FileIOfreqForces",
    "IMDfreqTime",
    "IMDfreqEnergies",
    "IMDfreqBox",
    "IMDfreqPositions",
    "IMDfreqVelocities",
    "IMDfreqForces",
    "PosWrap",
    "MDintstep",
    "nMDstep",
    "runTime",
    "performance",
    "stdevperformance"
]
engine_names = ["GROMACS", "NAMD", "LAMMPS"]
output_folder = "output"
data_file = "performance_data.csv"
all_df_list = []
for eng in engine_names:
    engine_path = os.path.join(eng, output_folder, data_file)
    engine_df = pd.read_csv(engine_path, sep=",", header=None)
    engine_df.columns = common_columns
    engine_df["MDengine"] = eng
    all_df_list.append(engine_df)
all_df = pd.concat(all_df_list, ignore_index=True)
all_df["performance"] = pd.to_numeric(all_df["performance"], errors='coerce')
all_df["nCPUused"] = pd.to_numeric(all_df["nCPUused"].astype(str).str.strip(), errors='coerce')
all_df["nCPUreq"] = pd.to_numeric(all_df["nCPUreq"].astype(str).str.strip(), errors='coerce')

## Optimization (Common Across Engines)

In [4]:
# Common optimization data preparation grouped by engine
opt_cpu_map = {
    "GROMACS": 24,
    "NAMD": 46,
    "LAMMPS": 8,
}
opt_df = all_df[all_df['SimType'].str.contains('vanilla|imdv3', case=False, na=False)].copy()
opt_df['SimTypeClean'] = opt_df['SimType'].str.strip().str.lower()
opt_grouped = opt_df.groupby(
    ["MDengine", "SimTypeClean", "nCPUused", "MPItasksperNode", "OpenMPthreadsperNode"],
    as_index=False
).agg(
    mean_performance=("performance", "mean"),
    stdev_performance=("performance", "std"),
    n_samples=("performance", "size")
)
opt_baseline_simtype = {
    "GROMACS": "imdv3",
    "NAMD": "imdv3",
    "LAMMPS": "imdv3",
}
opt_baseline = {}
for eng, cpu in opt_cpu_map.items():
    bsim = opt_baseline_simtype.get(eng, "vanilla")
    subset = opt_grouped[
        (opt_grouped["MDengine"] == eng)
        & (opt_grouped["SimTypeClean"].str.contains(bsim, na=False))
        & (opt_grouped["nCPUused"] == cpu)
    ]
    opt_baseline[eng] = subset["mean_performance"].max() if not subset.empty else np.nan

In [5]:
# Common optimization plotting (one output per engine)
engine_colors = {
    "GROMACS": ["blue", "red"],
    "NAMD": ["blue", "red"],
    "LAMMPS": ["blue", "red"],
}
label_map = {
    "vanilla": "pre-imdv3",
    "imdv3": "imdv3-modified",
}

for eng in engine_names:
    eng_opt = opt_grouped[opt_grouped["MDengine"] == eng]
    if eng_opt.empty:
        continue

    plt.figure()
    simtypes = ["vanilla", "imdv3"]
    colors_local = engine_colors.get(eng, ["blue", "red"])

    for sim_type in simtypes:
        group = eng_opt[eng_opt["SimTypeClean"].str.contains(sim_type, na=False)]
        if group.empty:
            continue

        color = colors_local[simtypes.index(sim_type)]
        display_label = label_map.get(sim_type, sim_type)

        # Simple marker-only plot, no error bars.
        plt.scatter(
            group["nCPUused"],
            group["mean_performance"],
            marker="o",
            s=12,
            facecolors="none",
            edgecolors=color,
            linewidths=0.8,
            label=display_label,
        )

    ylabel = "Performance (timesteps/s)" if eng == "LAMMPS" else "Performance (ns/day)"
    plt.xlabel("Number of cpu cores used")
    plt.ylabel(ylabel)
    plt.legend(loc="best")
    plt.tight_layout()
    plt.savefig(join(graphs_folder, f"FIG_{eng[:3]}_optimization.png"), dpi=1000, bbox_inches="tight")
    plt.show()

## Benchmarking (Common Across Engines)

In [6]:
# Common benchmarking data preparation grouped by engine
bench_df = all_df[
    all_df['SimType'].str.contains('fileio|streaming', case=False, na=False)
].copy()
bench_df['SimTypeClean'] = bench_df['SimType'].str.strip().str.lower()
bench_fileio_grouped = (
    bench_df[bench_df['SimTypeClean'].str.contains('fileio', na=False)]
    .groupby(['MDengine', 'SimTypeClean', 'FileIOfreqPositions'], as_index=False)
    .agg(mean_performance=('performance', 'mean'), std_performance=('performance', 'std'))
)
bench_stream_grouped = (
    bench_df[bench_df['SimTypeClean'].str.contains('streaming', na=False)]
    .groupby(['MDengine', 'SimTypeClean', 'IMDfreqPositions'], as_index=False)
    .agg(mean_performance=('performance', 'mean'), std_performance=('performance', 'std'))
)

In [7]:
# Common benchmarking plotting (one output per engine)
from scipy.optimize import least_squares
bench_palette_map = {
    "GROMACS": {"stream": colors_blue, "fileio": colors_green},
    "NAMD": {"stream": colors_red, "fileio": colors_orange},
    "LAMMPS": {"stream": colors_purple, "fileio": colors_teal},
}
# Marker/label mapping consistent across all engines
simtype_style = {
    "streaming": {"marker": "o", "label": "IMDv3-x streaming"},
    "streaming-3": {"marker": "s", "label": "IMDv3-xvf streaming"},
    "fileio": {"marker": "^", "label": "trr-x file I/O"},
    "fileio-3": {"marker": "D", "label": "trr-xvf file I/O"},
    "fileio-xtc": {"marker": "v", "label": "xtc-x file I/O"},
}
fileio_labels_by_engine = {
    "GROMACS": {"fileio": "trr-x file I/O", "fileio-3": "trr-xvf file I/O"},
    "NAMD": {"fileio": "dcd-x file I/O", "fileio-3": "dcd-xvf file I/O"},
    "LAMMPS": {"fileio": "lammpsdump-x file I/O", "fileio-3": "lammpsdump-xvf file I/O"},
}

def get_curve_style(engine, simtype):
    style = simtype_style.get(simtype, {"marker": "o", "label": simtype}).copy()
    if simtype in ("fileio", "fileio-3"):
        style["label"] = fileio_labels_by_engine.get(engine, {}).get(simtype, style["label"])
    return style

def lighten_color(color, amount=0.45):
    import matplotlib.colors as mcolors
    rgb = np.array(mcolors.to_rgb(color))
    return tuple(1 - (1 - rgb) * (1 - amount))

def _legend_save_show(savepath):
    handles, labels = plt.gca().get_legend_handles_labels()
    dedup = {}
    for h, l in zip(handles, labels):
        if l not in dedup:
            dedup[l] = h
    preferred_order = [
        "Optimal performance",
        "IMDv3-x streaming",
        "IMDv3-xvf streaming",
        "xtc-x file I/O",
        "trr-x file I/O",
        "trr-xvf file I/O",
        "dcd-x file I/O",
        "dcd-xvf file I/O",
        "lammpsdump-x file I/O",
        "lammpsdump-xvf file I/O",
    ]
    ordered_handles = [dedup[k] for k in preferred_order if k in dedup]
    ordered_labels = [k for k in preferred_order if k in dedup]
    for k, h in dedup.items():
        if k not in preferred_order:
            ordered_labels.append(k)
            ordered_handles.append(h)
    plt.legend(ordered_handles, ordered_labels, loc="lower right", frameon=False)
    plt.tight_layout()
    plt.savefig(savepath, dpi=1000, bbox_inches="tight")
    plt.show()

def fit_engine_global_models(eng, baseline, fit_candidate_curves, bench_df, fit_rows, fit_row_lookup):
    if np.isnan(baseline):
        return

    # Integration timestep scaling for speed model.
    dt_ns = 1.0
    dt_vals = pd.to_numeric(
        bench_df.loc[bench_df["MDengine"] == eng, "MDintstep"],
        errors="coerce",
    ).dropna()
    if eng == "NAMD" and not dt_vals.empty:
        dt_ns = float(dt_vals.median()) * 1e-6  # fs -> ns
    elif eng == "GROMACS" and not dt_vals.empty:
        dt_ns = float(dt_vals.median()) * 1e-3  # ps -> ns

    # Build curve specs from raw data, keeping individual runs.
    curve_specs = []
    seen_simtypes = set()
    for _, curve_color, curve_group, _, family in fit_candidate_curves:
        simtype = curve_group["SimTypeClean"].iloc[0]
        if simtype in seen_simtypes:
            continue
        seen_simtypes.add(simtype)

        freq_col = "IMDfreqPositions" if family == "streaming" else "FileIOfreqPositions"
        raw_curve = bench_df[
            (bench_df["MDengine"] == eng)
            & (bench_df["SimTypeClean"] == simtype)
        ][[freq_col, "performance"]].copy()

        raw_curve[freq_col] = pd.to_numeric(raw_curve[freq_col], errors="coerce")
        raw_curve["performance"] = pd.to_numeric(raw_curve["performance"], errors="coerce")
        raw_curve = raw_curve.replace([np.inf, -np.inf], np.nan).dropna()
        raw_curve = raw_curve[(raw_curve[freq_col] > 0) & (raw_curve["performance"] > 0)]
        if raw_curve.empty:
            continue

        freq_count = raw_curve.groupby(freq_col)[freq_col].transform("size").to_numpy(dtype=float)
        x_raw = raw_curve[freq_col].to_numpy(dtype=float)
        y_raw = raw_curve["performance"].to_numpy(dtype=float)
        n_freq = int(raw_curve[freq_col].nunique())
        if n_freq < 2:
            continue

        curve_specs.append({
            "simtype": simtype,
            "family": family,
            "color": curve_color,
            "x_raw": x_raw,
            "y_raw": y_raw,
            "w_freq": 1.0, # 1.0 / freq_count # Weight/normalize inversely proportional to number of runs at that freq, if needed
            "n_freq": n_freq,
        })

    if not curve_specs:
        return

    streaming_specs = [spec for spec in curve_specs if spec["family"] == "streaming"]
    fileio_specs = [spec for spec in curve_specs if spec["family"] == "fileio"]
    stream_idx = {spec["simtype"]: i for i, spec in enumerate(streaming_specs)}
    fileio_idx = {spec["simtype"]: i for i, spec in enumerate(fileio_specs)}

    n_stream = len(streaming_specs)
    n_fileio = len(fileio_specs)

    # Parameterization for model:
    # t_md global, one shared streaming t_c, per-streaming-curve t_w_i,
    # per-fileio-curve independent (t_c_i, t_w_i).
    i_t_md = 0
    i_t_const_stream = 1
    i_tws_stream_start = 2
    i_tconst_file_start = i_tws_stream_start + n_stream
    i_tws_file_start = i_tconst_file_start + n_fileio
    n_params = i_tws_file_start + n_fileio

    t_md0 = max(dt_ns / max(float(baseline), 1e-12), 1e-12)
    p0 = np.zeros(n_params, dtype=float)
    p0[i_t_md] = t_md0
    p0[i_t_const_stream] = 0.0

    for i, spec in enumerate(streaming_specs):
        freq1_mask = np.isclose(spec["x_raw"], 1.0)
        ref = np.mean(spec["y_raw"][freq1_mask]) if np.any(freq1_mask) else spec["y_raw"].max()
        p0[i_tws_stream_start + i] = max(dt_ns / max(ref, 1e-12) - t_md0, 0.0)

    for i, spec in enumerate(fileio_specs):
        freq1_mask = np.isclose(spec["x_raw"], 1.0)
        ref = np.mean(spec["y_raw"][freq1_mask]) if np.any(freq1_mask) else spec["y_raw"].max()
        p0[i_tconst_file_start + i] = 0.0
        p0[i_tws_file_start + i] = max(dt_ns / max(ref, 1e-12) - t_md0, 0.0)

    lower = np.zeros(n_params, dtype=float)
    upper = np.full(n_params, np.inf, dtype=float)

    def _curve_model(freq, t_md, t_c, t_w):
        return dt_ns / (t_md + t_c + t_w / freq)

    def residuals(params):
        t_md = params[i_t_md]
        out = []

        for spec in curve_specs:
            simtype = spec["simtype"]
            is_stream = 1.0 if spec["family"] == "streaming" else 0.0
            is_fileio = 1.0 if spec["family"] == "fileio" else 0.0

            tw_stream = params[i_tws_stream_start + stream_idx[simtype]] if simtype in stream_idx else 0.0
            tc_file = params[i_tconst_file_start + fileio_idx[simtype]] if simtype in fileio_idx else 0.0
            tw_file = params[i_tws_file_start + fileio_idx[simtype]] if simtype in fileio_idx else 0.0

            t_c = is_stream * params[i_t_const_stream] + is_fileio * tc_file
            t_w = is_stream * tw_stream + is_fileio * tw_file

            pred = _curve_model(spec["x_raw"], t_md, t_c, t_w)
            w_run = 1.0 # spec["w_freq"] / spec["n_freq"] # Weight/normalize inversely proportional to number of freqs sampled, if needed
            out.append((pred - spec["y_raw"]) * np.sqrt(w_run))

        out.append(np.array([(dt_ns / t_md) - baseline], dtype=float))
        return np.concatenate(out)

    try:
        fit = least_squares(
            residuals,
            p0,
            bounds=(lower, upper),
            max_nfev=50000,
        )
        params = fit.x
    except Exception:
        return

    t_md = params[i_t_md]

    for spec in curve_specs:
        simtype = spec["simtype"]
        is_stream = 1.0 if spec["family"] == "streaming" else 0.0
        is_fileio = 1.0 if spec["family"] == "fileio" else 0.0

        tw_stream = params[i_tws_stream_start + stream_idx[simtype]] if simtype in stream_idx else 0.0
        tc_file = params[i_tconst_file_start + fileio_idx[simtype]] if simtype in fileio_idx else 0.0
        tw_file = params[i_tws_file_start + fileio_idx[simtype]] if simtype in fileio_idx else 0.0

        t_c = is_stream * params[i_t_const_stream] + is_fileio * tc_file
        t_w = is_stream * tw_stream + is_fileio * tw_file

        plot_freq = np.logspace(np.log10(spec["x_raw"].min()), np.log10(spec["x_raw"].max()), 300)
        plot_speed = _curve_model(plot_freq, t_md, t_c, t_w)
        plt.plot(
            plot_freq,
            plot_speed,
            linestyle="-",
            color=lighten_color(spec["color"], amount=0.45),
            linewidth=1,
            label="_nolegend_",
        )

        row_idx = fit_row_lookup.get(simtype)
        if row_idx is not None:
            fit_rows[row_idx]["t_md"] = t_md
            fit_rows[row_idx]["t_const"] = t_c
            fit_rows[row_idx]["t_w"] = t_w

for eng in engine_names:
    eng_fileio = bench_fileio_grouped[bench_fileio_grouped["MDengine"] == eng]
    eng_stream = bench_stream_grouped[bench_stream_grouped["MDengine"] == eng]
    if eng_fileio.empty and eng_stream.empty:
        continue

    palette = bench_palette_map.get(eng, {"stream": colors_blue, "fileio": colors_green})
    baseline = opt_baseline.get(eng, np.nan)
    ylabel = "Performance (timesteps/s)" if eng == "LAMMPS" else "Performance (ns/day)"

    # Single main plot for all engines (includes GROMACS).
    model_simtypes = {"streaming", "streaming-3", "fileio-xtc", "fileio", "fileio-3"} if eng == "GROMACS" else None
    plt.figure(figsize=(fig_width, 0.775 * fig_width))
    fit_candidate_curves = []
    fit_rows = []
    fit_row_lookup = {}

    for simtype, group in eng_stream.groupby("SimTypeClean"):
        if model_simtypes is not None and simtype not in model_simtypes:
            continue
        style = get_curve_style(eng, simtype)
        shade = palette["stream"][1] if "streaming-3" in simtype else palette["stream"][0]
        plt.errorbar(
            group["IMDfreqPositions"],
            group["mean_performance"],
            yerr=group["std_performance"],
            marker=style["marker"],
            markersize=2,
            linestyle="None",
            capsize=2,
            label=style["label"],
            color=shade,
        )
        fit_candidate_curves.append((style["label"], shade, group.copy(), "IMDfreqPositions", "streaming"))
        fit_rows.append({
            "MDengine": eng,
            "curve_type": simtype,
            "curve_label": style["label"],
            "fit_family": "streaming",
            "fit_used": "yes",
            "t_md": "N/A",
            "t_const": "N/A",
            "t_w": "N/A",
        })
        fit_row_lookup[simtype] = len(fit_rows) - 1

    for simtype, group in eng_fileio.groupby("SimTypeClean"):
        if model_simtypes is not None and simtype not in model_simtypes:
            continue
        style = get_curve_style(eng, simtype)
        if "fileio-3" in simtype:
            shade = palette["fileio"][1]
        elif "xtc" in simtype:
            shade = palette["fileio"][2]
        else:
            shade = palette["fileio"][0]
        plt.errorbar(
            group["FileIOfreqPositions"],
            group["mean_performance"],
            yerr=group["std_performance"],
            marker=style["marker"],
            markersize=2,
            linestyle="None",
            capsize=2,
            label=style["label"],
            color=shade,
        )
        fit_candidate_curves.append((style["label"], shade, group.copy(), "FileIOfreqPositions", "fileio"))
        fit_rows.append({
            "MDengine": eng,
            "curve_type": simtype,
            "curve_label": style["label"],
            "fit_family": "fileio",
            "fit_used": "yes",
            "t_md": "N/A",
            "t_const": "N/A",
            "t_w": "N/A",
        })
        fit_row_lookup[simtype] = len(fit_rows) - 1

    if not np.isnan(baseline):
        plt.axhline(y=baseline, color="grey", linestyle="--", label="Optimal performance", linewidth=1)

    fit_engine_global_models(eng, baseline, fit_candidate_curves, bench_df, fit_rows, fit_row_lookup)

    plt.xlabel("streaming or file I/O interval")
    plt.ylabel(ylabel)
    plt.xscale("log")
    save_suffix = "benchmarking"
    _legend_save_show(join(graphs_folder, f"FIG_{eng[:3]}_{save_suffix}.png"))

    fit_param_table = pd.DataFrame(fit_rows)
    if eng == "LAMMPS":
        time_to_us = 1e6
    else:
        time_to_us = 86400.0 * 1e6
    for col in ["t_md", "t_const", "t_w"]:
        fit_param_table[col] = fit_param_table[col].apply(
            lambda val: val * time_to_us if isinstance(val, (int, float, np.floating)) else val
        )
        fit_param_table[col] = fit_param_table[col].apply(
            lambda val: f"{val:.6g}" if isinstance(val, (int, float, np.floating)) else val
        )
    print(f"Fit parameters for {eng}")
    display(fit_param_table)


Fit parameters for GROMACS


,MDengine,curve_type,curve_label,fit_family,fit_used,t_md,t_const,t_w
0,GROMACS,streaming,IMDv3-x streaming,streaming,yes,282.223,7.79387,8.39617
1,GROMACS,streaming-3,IMDv3-xvf streaming,streaming,yes,282.223,7.79387,473.45
2,GROMACS,fileio,trr-x file I/O,fileio,yes,282.223,72.7017,3721.31
3,GROMACS,fileio-3,trr-xvf file I/O,fileio,yes,282.223,90.693,7357.64
4,GROMACS,fileio-xtc,xtc-x file I/O,fileio,yes,282.223,17.6042,1366.42


Fit parameters for NAMD


,MDengine,curve_type,curve_label,fit_family,fit_used,t_md,t_const,t_w
0,NAMD,streaming,IMDv3-x streaming,streaming,yes,992.729,23.9234,2573.97
1,NAMD,streaming-3,IMDv3-xvf streaming,streaming,yes,992.729,23.9234,8088.87
2,NAMD,fileio,dcd-x file I/O,fileio,yes,992.729,1.58736,3797
3,NAMD,fileio-3,dcd-xvf file I/O,fileio,yes,992.729,2.65742,11454.4


Fit parameters for LAMMPS


,MDengine,curve_type,curve_label,fit_family,fit_used,t_md,t_const,t_w
0,LAMMPS,streaming,IMDv3-x streaming,streaming,yes,602.401,68.9094,1323.71
1,LAMMPS,streaming-3,IMDv3-xvf streaming,streaming,yes,602.401,68.9094,3980.71
2,LAMMPS,fileio,lammpsdump-x file I/O,fileio,yes,602.401,31.7515,21253.3


## Manuscript-specific composite figures (LAMMPS, NAMD and GROMACS)

In [8]:
# Shared helpers for manuscript benchmarking figures
from os.path import exists, join
from scipy.optimize import curve_fit, least_squares
import matplotlib.image as mpimg
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
import matplotlib.pyplot as plt
import numpy as np

def lighten_color(color, amount=0.45):
    import matplotlib.colors as mcolors
    rgb = np.array(mcolors.to_rgb(color))
    return tuple(1 - (1 - rgb) * (1 - amount))

def get_curve_style_from_map(simtype, style_map):
    return style_map.get(simtype, {"marker": "o", "label": simtype}).copy()

def plot_engine_points_with_errorbars(
    ax,
    eng,
    palette,
    style_map,
    bench_stream_grouped,
    bench_fileio_grouped,
    model_simtypes=None,
    fileio_fit_selector=None,
):
    eng_stream = bench_stream_grouped[bench_stream_grouped["MDengine"] == eng]
    eng_fileio = bench_fileio_grouped[bench_fileio_grouped["MDengine"] == eng]
    fit_candidate_curves = []

    for simtype, group in eng_stream.groupby("SimTypeClean"):
        if model_simtypes is not None and simtype not in model_simtypes:
            continue
        style = get_curve_style_from_map(simtype, style_map)
        shade = palette["stream"][1] if "streaming-3" in simtype else palette["stream"][0]
        ax.errorbar(
            group["IMDfreqPositions"],
            group["mean_performance"],
            yerr=group["std_performance"],
            marker=style["marker"],
            markersize=2,
            linestyle="None",
            capsize=2,
            label=style["label"],
            color=shade,
        )
        fit_candidate_curves.append((style["label"], shade, group.copy(), "IMDfreqPositions", "streaming"))

    for simtype, group in eng_fileio.groupby("SimTypeClean"):
        if model_simtypes is not None and simtype not in model_simtypes:
            continue
        style = get_curve_style_from_map(simtype, style_map)
        if "fileio-3" in simtype:
            shade = palette["fileio"][1]
        elif "xtc" in simtype:
            shade = palette["fileio"][2]
        else:
            shade = palette["fileio"][0]
        ax.errorbar(
            group["FileIOfreqPositions"],
            group["mean_performance"],
            yerr=group["std_performance"],
            marker=style["marker"],
            markersize=2,
            linestyle="None",
            capsize=2,
            label=style["label"],
            color=shade,
        )

        should_fit = True if fileio_fit_selector is None else fileio_fit_selector(simtype)
        if should_fit:
            fit_candidate_curves.append((style["label"], shade, group.copy(), "FileIOfreqPositions", "fileio"))

    return fit_candidate_curves

def plot_model_fit_lines(ax, eng, baseline, fit_candidate_curves, bench_df):
    if np.isnan(baseline):
        return

    # Integration timestep scaling for speed model.
    dt_ns = 1.0
    dt_vals = pd.to_numeric(
        bench_df.loc[bench_df["MDengine"] == eng, "MDintstep"],
        errors="coerce",
    ).dropna()
    if eng == "NAMD" and not dt_vals.empty:
        dt_ns = float(dt_vals.median()) * 1e-6  # fs -> ns
    elif eng == "GROMACS" and not dt_vals.empty:
        dt_ns = float(dt_vals.median()) * 1e-3  # ps -> ns

    # Build per-curve specs from raw runs.
    curve_specs = []
    seen_simtypes = set()
    for _, curve_color, curve_group, _, family in fit_candidate_curves:
        simtype = curve_group["SimTypeClean"].iloc[0]
        if simtype in seen_simtypes:
            continue
        seen_simtypes.add(simtype)

        freq_col = "IMDfreqPositions" if family == "streaming" else "FileIOfreqPositions"
        raw_curve = bench_df[
            (bench_df["MDengine"] == eng)
            & (bench_df["SimTypeClean"] == simtype)
        ][[freq_col, "performance"]].copy()

        raw_curve[freq_col] = pd.to_numeric(raw_curve[freq_col], errors="coerce")
        raw_curve["performance"] = pd.to_numeric(raw_curve["performance"], errors="coerce")
        raw_curve = raw_curve.replace([np.inf, -np.inf], np.nan).dropna()
        raw_curve = raw_curve[(raw_curve[freq_col] > 0) & (raw_curve["performance"] > 0)]
        if raw_curve.empty:
            continue

        freq_count = raw_curve.groupby(freq_col)[freq_col].transform("size").to_numpy(dtype=float)
        x_raw = raw_curve[freq_col].to_numpy(dtype=float)
        y_raw = raw_curve["performance"].to_numpy(dtype=float)
        n_freq = int(raw_curve[freq_col].nunique())
        if n_freq < 2:
            continue

        curve_specs.append({
            "simtype": simtype,
            "family": family,
            "color": curve_color,
            "x_raw": x_raw,
            "y_raw": y_raw,
            "w_freq": 1.0, # 1.0 / freq_count # Weight/normalize inversely proportional to number of runs at that freq, if needed
            "n_freq": n_freq,
        })

    if not curve_specs:
        return

    streaming_specs = [spec for spec in curve_specs if spec["family"] == "streaming"]
    fileio_specs = [spec for spec in curve_specs if spec["family"] == "fileio"]
    stream_idx = {spec["simtype"]: i for i, spec in enumerate(streaming_specs)}
    fileio_idx = {spec["simtype"]: i for i, spec in enumerate(fileio_specs)}

    n_stream = len(streaming_specs)
    n_fileio = len(fileio_specs)

    # Parameterization for model:
    # t_md global, one shared streaming t_c, per-streaming-curve t_w,
    # per-fileio-curve independent (t_c_i, t_w_i).
    i_t_md = 0
    i_t_const_stream = 1
    i_tws_stream_start = 2
    i_tconst_file_start = i_tws_stream_start + n_stream
    i_tws_file_start = i_tconst_file_start + n_fileio
    n_params = i_tws_file_start + n_fileio

    t_md0 = max(dt_ns / max(float(baseline), 1e-12), 1e-12)
    p0 = np.zeros(n_params, dtype=float)
    p0[i_t_md] = t_md0
    p0[i_t_const_stream] = 0.0

    for i, spec in enumerate(streaming_specs):
        freq1_mask = np.isclose(spec["x_raw"], 1.0)
        ref = np.mean(spec["y_raw"][freq1_mask]) if np.any(freq1_mask) else spec["y_raw"].max()
        p0[i_tws_stream_start + i] = max(dt_ns / max(ref, 1e-12) - t_md0, 0.0)

    for i, spec in enumerate(fileio_specs):
        freq1_mask = np.isclose(spec["x_raw"], 1.0)
        ref = np.mean(spec["y_raw"][freq1_mask]) if np.any(freq1_mask) else spec["y_raw"].max()
        p0[i_tconst_file_start + i] = 0.0
        p0[i_tws_file_start + i] = max(dt_ns / max(ref, 1e-12) - t_md0, 0.0)

    lower = np.zeros(n_params, dtype=float)
    upper = np.full(n_params, np.inf, dtype=float)

    def _curve_model(freq, t_md, t_c, t_w):
        return dt_ns / (t_md + t_c + t_w / freq)

    def residuals(params):
        t_md = params[i_t_md]
        out = []

        for spec in curve_specs:
            simtype = spec["simtype"]
            is_stream = 1.0 if spec["family"] == "streaming" else 0.0
            is_fileio = 1.0 if spec["family"] == "fileio" else 0.0

            tw_stream = params[i_tws_stream_start + stream_idx[simtype]] if simtype in stream_idx else 0.0
            tc_file = params[i_tconst_file_start + fileio_idx[simtype]] if simtype in fileio_idx else 0.0
            tw_file = params[i_tws_file_start + fileio_idx[simtype]] if simtype in fileio_idx else 0.0

            t_c = is_stream * params[i_t_const_stream] + is_fileio * tc_file
            t_w = is_stream * tw_stream + is_fileio * tw_file

            pred = _curve_model(spec["x_raw"], t_md, t_c, t_w)
            w_run = 1.0 # spec["w_freq"] / spec["n_freq"] # Weight/normalize inversely proportional to number of freqs sampled, if needed
            out.append((pred - spec["y_raw"]) * np.sqrt(w_run))

        out.append(np.array([(dt_ns / t_md) - baseline], dtype=float))
        return np.concatenate(out)

    try:
        fit = least_squares(
            residuals,
            p0,
            bounds=(lower, upper),
            max_nfev=50000,
        )
        params = fit.x
    except Exception:
        return

    t_md = params[i_t_md]

    for spec in curve_specs:
        simtype = spec["simtype"]
        is_stream = 1.0 if spec["family"] == "streaming" else 0.0
        is_fileio = 1.0 if spec["family"] == "fileio" else 0.0

        tw_stream = params[i_tws_stream_start + stream_idx[simtype]] if simtype in stream_idx else 0.0
        tc_file = params[i_tconst_file_start + fileio_idx[simtype]] if simtype in fileio_idx else 0.0
        tw_file = params[i_tws_file_start + fileio_idx[simtype]] if simtype in fileio_idx else 0.0

        t_c = is_stream * params[i_t_const_stream] + is_fileio * tc_file
        t_w = is_stream * tw_stream + is_fileio * tw_file

        plot_freq = np.logspace(np.log10(spec["x_raw"].min()), np.log10(spec["x_raw"].max()), 300)
        plot_speed = _curve_model(plot_freq, t_md, t_c, t_w)
        ax.plot(
            plot_freq,
            plot_speed,
            linestyle="-",
            color=lighten_color(spec["color"], amount=0.45),
            linewidth=1,
            label="_nolegend_",
        )

def apply_ordered_legend(ax, preferred_order, loc="lower right", frameon=False, ncols=1):
    handles, labels = ax.get_legend_handles_labels()
    dedup = {}
    for h, l in zip(handles, labels):
        if l not in dedup:
            dedup[l] = h
    ordered_labels = [k for k in preferred_order if k in dedup]
    ordered_handles = [dedup[k] for k in ordered_labels]
    for k, h in dedup.items():
        if k not in ordered_labels:
            ordered_labels.append(k)
            ordered_handles.append(h)
    ax.legend(ordered_handles, ordered_labels, loc=loc, frameon=frameon, ncols=ncols)

def add_image_inset(ax, image_path, zoom, xy):
    if exists(image_path):
        img = mpimg.imread(image_path)
        imagebox = OffsetImage(img, zoom=zoom)
        ab = AnnotationBbox(
            imagebox,
            xy,
            xycoords="axes fraction",
            frameon=False,
            box_alignment=(0.5, 0.5),
            zorder=5,
        )
        ax.add_artist(ab)

# Per-engine config: palette colors, marker/label styles, fit options, image insets
ENGINE_CONFIG = {
    "GROMACS": {
        "palette": {"stream": colors_blue, "fileio": colors_green},
        "simtype_style": {
            "streaming":   {"marker": "o", "label": "IMDv3-x streaming"},
            "streaming-3": {"marker": "s", "label": "IMDv3-xvf streaming"},
            "fileio":      {"marker": "^", "label": "trr-x file I/O"},
            "fileio-3":    {"marker": "D", "label": "trr-xvf file I/O"},
            "fileio-xtc":  {"marker": "v", "label": "xtc-x file I/O"},
        },
        "model_simtypes": {"streaming", "streaming-3", "fileio-xtc", "fileio", "fileio-3"},
        "fileio_fit_selector": lambda simtype: "xtc" in simtype,
        "preferred_order": [
            "Optimal performance", "IMDv3-x streaming", "IMDv3-xvf streaming",
            "xtc-x file I/O", "trr-x file I/O", "trr-xvf file I/O",
        ],
        "ylabel": "Performance (ns/day)",
        "img_path": join("GROMACS", "input", "HEWL_benchmark_system.bmp"),
        "img_zoom": 0.042,
        "img_xy": (0.848, 0.42),
        "legend_ncols": 2,
        "xlim_right": 4e5,
        "savefile": "FIG_GRO_benchmarking_inset.png",
    },
    "LAMMPS": {
        "palette": {"stream": colors_purple, "fileio": colors_teal},
        "simtype_style": {
            "streaming":   {"marker": "o", "label": "IMDv3-x streaming"},
            "streaming-3": {"marker": "s", "label": "IMDv3-xvf streaming"},
            "fileio":      {"marker": "^", "label": "lammpsdump-x file I/O"},
            "fileio-3":    {"marker": "D", "label": "lammpsdump-xvf file I/O"},
        },
        "model_simtypes": None,
        "fileio_fit_selector": None,
        "preferred_order": [
            "Optimal performance", "IMDv3-x streaming", "IMDv3-xvf streaming",
            "lammpsdump-x file I/O", "lammpsdump-xvf file I/O", "xtc-x file I/O",
        ],
        "ylabel": "Performance (timesteps/s)",
        "img_path": join("LAMMPS", "input", "poly-chain_benchmarking_system.bmp"),
        "img_zoom": 0.05,
        "img_xy": (0.8, 0.49),
        "legend_ncols": 1,
        "xlim_right": None,
        "savefile": "FIG_LAM_benchmarking_inset.png",
    },
    "NAMD": {
        "palette": {"stream": colors_red, "fileio": colors_orange},
        "simtype_style": {
            "streaming":   {"marker": "o", "label": "IMDv3-x streaming"},
            "streaming-3": {"marker": "s", "label": "IMDv3-xvf streaming"},
            "fileio":      {"marker": "^", "label": "dcd-x file I/O"},
            "fileio-3":    {"marker": "D", "label": "dcd-xvf file I/O"},
        },
        "model_simtypes": None,
        "fileio_fit_selector": None,
        "preferred_order": [
            "Optimal performance", "IMDv3-x streaming", "IMDv3-xvf streaming",
            "dcd-x file I/O", "dcd-xvf file I/O", "xtc-x file I/O",
        ],
        "ylabel": "Performance (ns/day)",
        "img_path": join("GROMACS", "input", "HEWL_benchmark_system.bmp"), # same system as GROMACS
        "img_zoom": 0.058,
        "img_xy": (0.79, 0.52),
        "legend_ncols": 2,
        "xlim_right": None,
        "savefile": "FIG_NAM_benchmarking_inset.png",
    },
}

In [9]:
# Manuscript figure: GROMACS benchmarking models plot with HEWL image inside plot
eng = "GROMACS"
cfg = ENGINE_CONFIG[eng]
baseline = opt_baseline.get(eng, np.nan)

fig, ax = plt.subplots(figsize=(fig_width * 1.08, 0.80 * fig_width))

fit_candidate_curves = plot_engine_points_with_errorbars(
    ax=ax, eng=eng,
    palette=cfg["palette"], style_map=cfg["simtype_style"],
    bench_stream_grouped=bench_stream_grouped, bench_fileio_grouped=bench_fileio_grouped,
    model_simtypes=cfg["model_simtypes"], fileio_fit_selector=cfg["fileio_fit_selector"],
)

if not np.isnan(baseline):
    ax.axhline(y=baseline, color="grey", linestyle="--", label="Optimal performance", linewidth=1)

plot_model_fit_lines(ax=ax, eng=eng, baseline=baseline, fit_candidate_curves=fit_candidate_curves, bench_df=bench_df)

apply_ordered_legend(ax, cfg["preferred_order"], loc="lower right", frameon=False, ncols=cfg["legend_ncols"])
add_image_inset(ax, cfg["img_path"], zoom=cfg["img_zoom"], xy=cfg["img_xy"])

ax.set_xlabel("streaming or file I/O interval")
ax.set_ylabel(cfg["ylabel"])
ax.set_xscale("log")
if cfg["xlim_right"] is not None:
    ax.set_xlim(right=cfg["xlim_right"])
fig.tight_layout()
fig.savefig(join(graphs_folder, cfg["savefile"]), dpi=1000, bbox_inches="tight")
plt.show()


In [10]:
# Manuscript figure: LAMMPS benchmarking plot with polymer image inside plot
eng = "LAMMPS"
cfg = ENGINE_CONFIG[eng]
baseline = opt_baseline.get(eng, np.nan)

fig, ax = plt.subplots(figsize=(fig_width * 1.08, 0.80 * fig_width))

fit_candidate_curves = plot_engine_points_with_errorbars(
    ax=ax, eng=eng,
    palette=cfg["palette"], style_map=cfg["simtype_style"],
    bench_stream_grouped=bench_stream_grouped, bench_fileio_grouped=bench_fileio_grouped,
    model_simtypes=cfg["model_simtypes"], fileio_fit_selector=cfg["fileio_fit_selector"],
)

if not np.isnan(baseline):
    ax.axhline(y=baseline, color="grey", linestyle="--", label="Optimal performance", linewidth=1)

plot_model_fit_lines(ax=ax, eng=eng, baseline=baseline, fit_candidate_curves=fit_candidate_curves, bench_df=bench_df)

apply_ordered_legend(ax, cfg["preferred_order"], loc="lower right", frameon=False, ncols=cfg["legend_ncols"])
add_image_inset(ax, cfg["img_path"], zoom=cfg["img_zoom"], xy=cfg["img_xy"])

ax.set_xlabel("streaming or file I/O interval")
ax.set_ylabel(cfg["ylabel"])
ax.set_xscale("log")
if cfg["xlim_right"] is not None:
    ax.set_xlim(right=cfg["xlim_right"])
fig.tight_layout()
fig.savefig(join(graphs_folder, cfg["savefile"]), dpi=1000, bbox_inches="tight")
plt.show()


In [11]:
# Manuscript figure: NAMD benchmarking models plot with HEWL image inside plot
eng = "NAMD"
cfg = ENGINE_CONFIG[eng]
baseline = opt_baseline.get(eng, np.nan)

fig, ax = plt.subplots(figsize=(fig_width * 1.08, 0.80 * fig_width))

fit_candidate_curves = plot_engine_points_with_errorbars(
    ax=ax, eng=eng,
    palette=cfg["palette"], style_map=cfg["simtype_style"],
    bench_stream_grouped=bench_stream_grouped, bench_fileio_grouped=bench_fileio_grouped,
    model_simtypes=cfg["model_simtypes"], fileio_fit_selector=cfg["fileio_fit_selector"],
)

if not np.isnan(baseline):
    ax.axhline(y=baseline, color="grey", linestyle="--", label="Optimal performance", linewidth=1)

plot_model_fit_lines(ax=ax, eng=eng, baseline=baseline, fit_candidate_curves=fit_candidate_curves, bench_df=bench_df)

apply_ordered_legend(ax, cfg["preferred_order"], loc="lower right", frameon=False, ncols=cfg["legend_ncols"])
add_image_inset(ax, cfg["img_path"], zoom=cfg["img_zoom"], xy=cfg["img_xy"])

ax.set_xlabel("streaming or file I/O interval")
ax.set_ylabel(cfg["ylabel"])
ax.set_xscale("log")
if cfg["xlim_right"] is not None:
    ax.set_xlim(right=cfg["xlim_right"])
fig.tight_layout()
fig.savefig(join(graphs_folder, cfg["savefile"]), dpi=1000, bbox_inches="tight")
plt.show()
